# 19 — MCP Server Policies and Governance

Every policy you attach to an MCP server in API Management's **MCP >
Policies** blade applies to *all* of its exposed tools -- there's no
per-tool policy scope yet. This notebook walks through the governance
policies you'll reach for most often, drawn from
[Configure policies for the MCP server](https://learn.microsoft.com/en-us/azure/api-management/export-rest-mcp-server#configure-policies-for-the-mcp-server).

Policies are evaluated in this order: global (all-APIs) scope first, then
the MCP server's own scope.

## Rate limiting and quotas

Cap how often a client (or IP) can call any tool on this MCP server:

```xml
<inbound>
    <base />
    <rate-limit-by-key
        calls="5"
        renewal-period="30"
        counter-key="@(context.Request.IpAddress)"
        remaining-calls-variable-name="remainingCallsPerIP" />
</inbound>
```

Pair this with `quota-by-key` if you also want a longer-window budget (for
example, a daily cap per subscription) in addition to the short-window
rate limit above.

## IP filtering and response caching

Restrict which networks can reach the tools at all:

```xml
<inbound>
    <base />
    <ip-filter action="allow">
        <address-range from="203.0.113.0" to="203.0.113.255" />
    </ip-filter>
</inbound>
```

For tools that are expensive but idempotent (read-only lookups, for
example), cache their responses to cut backend load and latency:

```xml
<inbound>
    <base />
    <cache-lookup vary-by-developer="false" vary-by-developer-groups="false" />
</inbound>
<outbound>
    <base />
    <cache-store duration="60" />
</outbound>
```

Only cache tools whose results are safe to reuse across callers for the
configured duration -- never cache a tool that returns per-user or
time-sensitive data this way.

## Tracing and observability

Add a custom trace so tool calls show up with useful metadata in the test
console, Application Insights, and resource logs:

```xml
<inbound>
    <base />
    <trace source="My MCP" severity="information">
        <message>My MCP trace info</message>
        <metadata name="agent-id" value="@(context.Request.Headers.GetValueOrDefault(&quot;agent-id&quot;, &quot;n/a&quot;))" />
    </trace>
</inbound>
```

This is the same [`trace` policy](https://learn.microsoft.com/en-us/azure/api-management/trace-policy)
used elsewhere in API Management -- nothing MCP-specific about it, which is
exactly the point: MCP servers inherit the whole API Management policy
toolbox.